# Concurrency and Asynchronous Workflows
This notebook covers the conceptual differences between **multi-threading and multi-processing**, the motive behind **asynchronous programming**, and the role of **event loops in non-blocking systems**. In addition to the implementation of  **asynchronous I/O programs using asyncio** and **concurrent HTTP requests using aiohttp** . Finally,  the role of **task queues and scheduling systems**. Each section includes explanations and examples.

## Concurrency Matters

Modern software systems must handle:

- large numbers of requests
- network communication
- disk access
- long-running computations

Sequential execution often wastes time because programs frequently **wait for external resources**.

Typical waiting operations include:

- file reading
- network requests
- database queries
- API calls


Concurrency techniques allow programs to **perform useful work while waiting**.

Three main **concurrency approaches** exist:

- Multi-threading

- Multi-processing

- Asynchronous programming

Each approach addresses **different system constraints**.

   

# 1. Multi-threading

A **thread** is a lightweight execution unit within a process.

Characteristics:

- shares the same memory space
- cheaper to create than processes
- suitable for **I/O-bound tasks** (network, disk, waiting)

However, Python has an **important limitation** 

### The Global Interpreter Lock (GIL)

The GIL ensures that **only one thread executes Python bytecode at a time**.

Implication:

- Multi-threading does **not improve CPU-bound performance**
- It is still effective for **I/O-bound workloads**

Think of it like this: Threads are like  **Multiple chefs** sharing **ONE knife** due to (GIL).


>Before running the bellow cells, have a look at [threading documentation](https://docs.python.org/3/library/threading.html) and 
[time documentation](https://docs.python.org/3/library/time.html)

In [ ]:
# Create and run multiple threads to execute tasks concurrently.
# Each task simulates an I/O-bound operation, while join() ensures the main thread waits for all tasks to complete.
import threading
import time

def task(name):
    print(f"\nStarting {name}")
    time.sleep(2)  # releases the GIL, other threads can run during this wait
    print(f"\nFinished {name}")

threads = []

for i in range(3):
    t = threading.Thread(target=task, args=(f"Task-{i}",))
    threads.append(t)
    t.start() # start thread execution

for t in threads:
    t.join()   # wait for all threads to finish


In [ ]:
import time
import threading

def io_task(task_id, sleep_duration=1.0):
    time.sleep(sleep_duration)  # GIL is released here, allowing other threads to run
    print(f"  Task {task_id} done")

def demo_threading_io():
    start = time.perf_counter() # returns a high-resolution timer value used to measure how long the code runs
    threads = [threading.Thread(target=io_task, args=(i,)) for i in range(5)] # Create 5 threads, each running io_task with a different task ID
    for t in threads: t.start()
    for t in threads: t.join()
    elapsed = time.perf_counter() - start
    print(f"[Threading  | I/O] {elapsed:.2f}s  (5 x 1s tasks)")

def demo_sequential_io():
    start = time.perf_counter()
    for i in range(5):
        io_task(i)
    elapsed = time.perf_counter() - start
    print(f"[Sequential | I/O] {elapsed:.2f}s  (5 x 1s tasks)")

if __name__ == "__main__":
    # Compares sequential execution against multithreaded execution for I/O-bound tasks to highlight concurrency benefits.
    print("Running sequential I/O tasks:")
    demo_sequential_io()  # expected: ~5s

    print("\nRunning threaded I/O tasks:")
    demo_threading_io()   # expected: ~1s, all 5 sleep at the same time

# 2. Multi-Processing

A **process** is an independent program instance with its own memory space.

Unlike threads:

- processes do not share memory

- processes bypass the GIL

- processes execute on multiple CPU cores

This makes multi-processing ideal for CPU-bound computations.

Examples:

- numerical simulations

- machine learning training

- large mathematical calculations

Think of it like this: Processes are like **Multiple chefs**, each with their **OWN knife** (separate memory).


> Before running the below cells, have a look at [multiprocessing documentation](https://docs.python.org/3/library/multiprocessing.html).

### Multi-Processing Execution Flow
> Check & run [test_mp.py](test_mp.py) and have a look at the execution flow.
Each process executes independently.
```
Main Program
      │
      ▼
Create Processes
      │
      ├─────────────────┬─────────────────┐
      ▼                 ▼                 ▼
 Process 1          Process 2          Process 3
      │                 │                 │
      ▼                 ▼                 ▼
 Computation        Computation        Computation
 (CPU Core 1)       (CPU Core 2)       (CPU Core 3)
      │                 │                 │
      └─────────────────┴─────────────────┘
                        │
                        ▼
              Main waits via join()
                        │
                        ▼
               All Processes Done
```

> Because processes run on **separate CPU cores**, they achieve **true parallelism**.  Unlike threads, which are limited by Python's GIL.

### Multi-threading vs Multi-processing

In [ ]:
import threading
import multiprocessing
import time

def cpu_heavy(n):
    total = 0
    for i in range(n):
        total += i * i  # pure CPU work, GIL is never released during this loop
    return total

def demo_threading_cpu():
    # Two threads sharing one GIL, so they take turns instead of running in parallel
    start = time.perf_counter()
    t1 = threading.Thread(target=cpu_heavy, args=(10_000_000,))
    t2 = threading.Thread(target=cpu_heavy, args=(10_000_000,))
    t1.start(); t2.start()
    t1.join();  t2.join()
    print(f"[Threading  | CPU] {time.perf_counter() - start:.2f}s")

def demo_multiprocessing_cpu():
    # Two separate processes, each with its own interpreter, so they truly run in parallel
    start = time.perf_counter()
    p1 = multiprocessing.Process(target=cpu_heavy, args=(10_000_000,))
    p2 = multiprocessing.Process(target=cpu_heavy, args=(10_000_000,))
    p1.start(); p2.start()
    p1.join();  p2.join()
    print(f"[Processing | CPU] {time.perf_counter() - start:.2f}s")

if __name__ == "__main__":
    print("Running threading tasks:")
    demo_threading_cpu()

    print("\nRunning processing tasks:")
    demo_multiprocessing_cpu()


# 3. Asynchronous Programming 

Threads and processes can handle concurrency, but they introduce overhead:

- thread context switching (threads do not run in true parallel for CPU-bound code due to the GIL)

- process memory duplication

Synchronous:   Task A → wait → Task A done → Task B → wait → Task B done

Asynchronous:  Task A starts → (while waiting) → Task B starts → Task A done → Task B done

For applications with many network operations, asynchronous programming is more efficient.

Examples:

- web servers

- chat systems

- API aggregators

- web crawlers

Four concepts define asynchronous programming.

1. **Coroutine**

A coroutine is a cooperatively scheduled function defined with:
```python
async def function_name():
```
2. **Await**

The ```await``` keyword pauses execution until an operation completes.

3. **Event Loop**

The event loop manages all asynchronous tasks and schedules their execution, switching between them on ```await```.

4. **Task**
A coroutine scheduled to run concurrently on the event loop

> Before running the below cells, have a look at [asyncio documentation](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html)

In [ ]:
# asyncio is Python's built-in library for writing concurrent code using coroutines.
# It provides the event loop, which acts as the central scheduler that decides
# which coroutine runs, pauses, and resumes at any given moment.
import asyncio

# `async def` marks this as a coroutine function. It does NOT run immediately when called.
# Instead, calling greet() returns a coroutine OBJECT that must be awaited or scheduled
# before any of its body executes.
async def greet():

    # Executes immediately when the coroutine first runs 
    # no await here means no pause, no yielding to the event loop
    print("Hello")

    # `await` does two things simultaneously:
    #   1. Suspends this coroutine for 2 seconds (it stops here and waits)
    #   2. Hands control back to the event loop so other tasks can run in the meantime
    # This is fundamentally different from time.sleep(2), which would freeze
    # the entire program, nothing else could execute during that block
    await asyncio.sleep(2)

    # The event loop resumes execution HERE once the 2-second timer fires.
    # Any other tasks that were scheduled would have had their chance to run
    # during those 2 seconds before we reach this line.
    print("World")


# ── Jupyter-specific execution ──────────────────────────────────────────────
# Jupyter notebooks already run an active event loop in the background.
# asyncio.run() would raise an error here because it tries to START a new loop
# on top of the existing one, which is not allowed.
#
# Instead, Jupyter lets you `await` coroutines directly inside notebook cells,
# handing the coroutine straight to the already-running event loop.
#
# Outside Jupyter (e.g. a .py script), replace this line with:
#   asyncio.run(greet())
await greet()  # Jupyter directly awaits the coroutine in the existing event loop


# Coroutine Execution Flow `greet()`
```
greet() starts
      │
      ▼
prints "Hello"
      │
      ▼
await asyncio.sleep(2)
      │
      ▼
coroutine pauses ──────────────► event loop is free
                                  to run other tasks
      │                                   │
      │        (2 seconds pass)           │
      │                                   │
      ◄──────────────────────────────────┘
      │
      ▼
execution resumes
      │
      ▼
prints "World"
      │
      ▼
  coroutine ends
```

> `await` does **not** block the thread, it suspends only the coroutine,
> handing control back to the **event loop** until the awaited task completes.

In [ ]:
# ─────────────────────────────────────────────
# This allows tasks A, B, and C to run concurrently rather than sequentially,
# making the code more efficient by not blocking while waiting for each task to complete.
# Even though each task waits 2 seconds, the total runtime is ~2 seconds
# because all three tasks overlap (they all await at the same time.)
# ─────────────────────────────────────────────
import asyncio

async def worker(name):
    print(f"Worker {name} started") # Signals this coroutine has begun executing

    # `await asyncio.sleep(2)` suspends this coroutine for 2 seconds
    # Crucially, it does NOT block the event loop, switches to other ready tasks
    # Compare to time.sleep(2), which would freeze the entire program
    await asyncio.sleep(2)  # yields control, other workers run during this wait
    print(f"Worker {name} finished")

async def main():
    # create_task schedules each coroutine immediately on the event loop
    task1 = asyncio.create_task(worker("A"))
    task2 = asyncio.create_task(worker("B"))
    task3 = asyncio.create_task(worker("C"))

    # gather waits for all three to finish, total time ~2s not 6s 
    # gather() suspends main() and waits until ALL three tasks finish
    await asyncio.gather(task1, task2, task3)

await main()


# Async Task Concurrency Flow  `main()`
```
main()
   │
   ▼
create tasks
   │
   ├──────────────────┬──────────────────┐
   ▼                  ▼                  ▼
 Task 1             Task 2             Task 3
   │                  │                  │
   ▼                  ▼                  ▼
await sleep(2)    await sleep(2)    await sleep(2)
   │                  │                  │
   └──────────────────┴──────────────────┘
                       │
                       ▼
          event loop schedules tasks
                       │
                       ▼
          tasks run concurrently
                       │
                       ▼
              all tasks complete
```

> **Total runtime ≈ 2 seconds**, not 6 seconds.
> All three tasks `await` simultaneously, so the event loop
> runs them **side by side** rather than one after another.

## 4. Async I/O with asyncio and aiohttp

Why aiohttp?

- requests (synchronous) → blocks the thread while waiting for the HTTP response
- aiohttp  (async)       → releases event loop while waiting → other coroutines run

When **fetching 100 URLs**:
  - requests sequential: 100 * avg_latency (e.g. 100 * 200ms = 20s)
  - aiohttp concurrent:  max_latency + overhead  (e.g. ~200ms + overhead = ~0.5s)


In [ ]:
pip install aiohttp

> Before running the following cells, have a look at [aiohttp documentation](https://docs.aiohttp.org/en/stable/).

In [ ]:
import asyncio
import aiohttp

# Coroutine: fetches a web page asynchronously without blocking the event loop
async def fetch_page(url):
    # Creates an asynchronous HTTP session (efficient connection management)
    async with aiohttp.ClientSession() as session:
        # Sends a non-blocking GET request
        async with session.get(url) as response:

            # waits for full response body without blocking other coroutines
            content = await response.text()

            # returns computed result (page size in characters)
            return len(content)

# Orchestrator coroutine: coordinates asynchronous execution
async def main():
    # awaits completion of the asynchronous HTTP request
    size = await fetch_page("https://example.com")

    # outputs final computed result
    print(f"Page size: {size} characters")

# runs the coroutine in the active event loop
await main()


## 5. Task Queues and Scheduling Fundamentals

In real systems, tasks are often processed using a producer-consumer model.

Architecture:

Producer → Task Queue → Worker → Result

> Before running the following cell, have a look at [queue documentation](https://docs.python.org/3/library/queue.html)

In [ ]:
# Demonstrates the producer-consumer pattern using a shared queue and worker threads.
# The producer adds tasks to the queue, while workers process them concurrently.

import threading
import queue
import time

def worker(task_queue):
    while True:
        task = task_queue.get()

        # None is used as a signal to stop the worker
        if task is None:
            task_queue.task_done()
            break

        print(f"\n Processing task {task}")
        time.sleep(1)  # simulate work
        print(f"\n Finished task {task}")

        task_queue.task_done()

# Shared task queue
task_queue = queue.Queue()

# Create and start worker threads
workers = []
for _ in range(2):
    thread = threading.Thread(target=worker, args=(task_queue,))
    thread.start()
    workers.append(thread)

# Producer: add tasks to the queue
for i in range(5):
    task_queue.put(i)

# Wait until all tasks have been processed
task_queue.join()

# Stop workers gracefully
for _ in workers:
    task_queue.put(None)

# Wait for all workers to exit
for thread in workers:
    thread.join()

print("\n All tasks completed")

### Now we are done with week 2's lab content 🎉🎉🎉 ##
![Week 2](Multi.png)
### Let's navigate to Exercise's notebook ⬇️⬇️⬇️
[Exercise Notebook](Week2_Exercise.ipynb)